# fNIRS 数据质量检查（QC Analysis）

**设备**：CRB35 近红外成像仪（院内设备）  
**任务**：视频观看期间大脑激活  
**数据格式**：Homer2 `.nirs` 格式（以 `.bin` 扩展名保存）  
**波长**：690 nm / 830 nm  
**采样率**：50 Hz

本 Notebook 执行以下检查：
1. 数据基本信息
2. 原始信号可视化
3. Scalp Coupling Index (SCI) — 头皮耦合质量
4. 坏道识别
5. 运动伪影检测
6. 刺激/辅助通道查看

## 1. 导入库 & 工具函数

In [ ]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'

# Homer2 .bin 加载函数
def load_nirs(filepath):
    mat = scipy.io.loadmat(str(filepath), appendmat=False)
    d   = mat['d'].astype(float)          # (n_times, n_ch)
    t   = mat['t'].flatten()              # (n_times,)
    ml  = mat['ml'].astype(int)           # (n_ch, 4): src, det, wl_idx, type
    s   = mat['s']                        # (n_times, n_stim)
    aux = mat['aux']                      # (n_times, n_aux)
    SD  = mat['SD']
    lambdas = SD['Lambda'][0, 0].flatten().tolist()  # [690, 830]
    
    sfreq = 1.0 / float(np.median(np.diff(t)))
    
    return dict(d=d, t=t, ml=ml, s=s, aux=aux,
                lambdas=lambdas, sfreq=sfreq, filepath=filepath)

# 数据路径
DATA_ROOT = Path('../')
SUBJECTS  = ['0', '1']

datasets = {}
for sub in SUBJECTS:
    files = list((DATA_ROOT / sub).glob('*.bin'))
    if files:
        datasets[sub] = [load_nirs(f) for f in sorted(files)]
        print(f"Subject {sub}: {len(files)} 个文件 — {[f.name for f in sorted(files)]}")
    else:
        print(f"Subject {sub}: 未找到文件")

## 2. 基本数据信息

In [ ]:
rows = []
for sub, runs in datasets.items():
    for run in runs:
        n_pairs = (run['ml'][:, 3] == 1).sum()  # ml col3 = wavelength index
        rows.append({
            '被试': f"Sub-{sub}",
            '文件': Path(run['filepath']).name,
            '采样率(Hz)': f"{run['sfreq']:.1f}",
            '总时长(min)': f"{run['t'][-1]/60:.1f}",
            '时间点数': run['d'].shape[0],
            '总通道数': run['d'].shape[1],
            'SD对数(WL1)': n_pairs,
            '波长(nm)': str(run['lambdas']),
            'Aux通道数': run['aux'].shape[1],
        })

pd.DataFrame(rows)

## 3. 原始信号可视化（前 60 秒）

In [ ]:
def plot_raw_signals(run, sub_label, duration_s=60, n_ch_show=20):
    d, t, ml = run['d'], run['t'], run['ml']
    mask_t = t <= duration_s
    t_seg  = t[mask_t]

    wl1_idx = np.where(ml[:, 3] == 1)[0][:n_ch_show]
    wl2_idx = np.where(ml[:, 3] == 2)[0][:n_ch_show]

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    lambdas = run['lambdas']

    for ax, idx, wl in zip(axes, [wl1_idx, wl2_idx], lambdas):
        seg = d[mask_t][:, idx]
        vmin = np.nanpercentile(seg, 1, axis=0)
        vmax = np.nanpercentile(seg, 99, axis=0)
        rng  = np.where(vmax - vmin > 0, vmax - vmin, 1)
        seg_norm = (seg - vmin) / rng

        offset = np.arange(len(idx))
        for i, ch in enumerate(idx):
            ax.plot(t_seg, seg_norm[:, i] + offset[i], lw=0.5,
                    color='steelblue' if wl == lambdas[0] else 'tomato', alpha=0.8)
        ax.set_yticks(offset)
        ax.set_yticklabels([f"S{ml[c,0]}-D{ml[c,1]}" for c in idx], fontsize=7)
        ax.set_title(f"{sub_label} | {wl} nm（前{n_ch_show}道）")
        ax.set_ylabel("通道（归一化偏移）")

    axes[-1].set_xlabel("时间 (s)")
    plt.suptitle(f"{sub_label} — 原始光强信号（前 {duration_s}s）", fontsize=12)
    plt.tight_layout()
    plt.show()

for sub, runs in datasets.items():
    for run in runs:
        plot_raw_signals(run, f"Sub-{sub} / {Path(run['filepath']).name}")

## 4. Scalp Coupling Index (SCI)

SCI = 两波长同一 S-D 对的 Pearson 相关系数（衡量头皮光耦合质量）。  
**SCI ≥ 0.75** → 通道质量良好；< 0.75 → 可能存在光路耦合问题。

In [ ]:
SCI_THRESHOLD = 0.75

def compute_sci(run):
    d, ml = run['d'], run['ml']
    wl1_rows = np.where(ml[:, 3] == 1)[0]  # col3 = wavelength index
    wl2_rows = np.where(ml[:, 3] == 2)[0]

    pairs_wl1 = {(ml[i, 0], ml[i, 1]): i for i in wl1_rows}
    pairs_wl2 = {(ml[i, 0], ml[i, 1]): i for i in wl2_rows}
    common = set(pairs_wl1) & set(pairs_wl2)

    sci_records = []
    for (src, det) in sorted(common):
        i1, i2 = pairs_wl1[(src, det)], pairs_wl2[(src, det)]
        x1 = d[:, i1]
        x2 = d[:, i2]
        valid = np.isfinite(x1) & np.isfinite(x2)
        if valid.sum() < 10:
            r = np.nan
        else:
            r = np.corrcoef(x1[valid], x2[valid])[0, 1]
        sci_records.append({'src': src, 'det': det,
                            'ch_label': f"S{src}-D{det}",
                            'SCI': r,
                            'good': bool(r >= SCI_THRESHOLD) if np.isfinite(r) else False})
    return pd.DataFrame(sci_records)

all_sci = {}
for sub, runs in datasets.items():
    for run in runs:
        key = f"Sub-{sub} / {Path(run['filepath']).name}"
        df  = compute_sci(run)
        all_sci[key] = df
        n_good = df['good'].sum()
        n_total = len(df)
        print(f"{key}:  {n_good}/{n_total} 通道 SCI>={SCI_THRESHOLD}  ({100*n_good/n_total:.0f}%)")

In [ ]:
def plot_sci(sci_df, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 柱状图
    colors = ['#2ecc71' if g else '#e74c3c' for g in sci_df['good']]
    axes[0].bar(range(len(sci_df)), sci_df['SCI'], color=colors, edgecolor='none')
    axes[0].axhline(SCI_THRESHOLD, color='k', ls='--', lw=1.5, label=f'阈值={SCI_THRESHOLD}')
    axes[0].set_xticks(range(len(sci_df)))
    axes[0].set_xticklabels(sci_df['ch_label'], rotation=90, fontsize=7)
    axes[0].set_ylabel('SCI')
    axes[0].set_ylim(-0.1, 1.05)
    axes[0].legend()
    axes[0].set_title('每对 S-D 的 SCI 值')

    # 直方图
    valid_sci = sci_df['SCI'].dropna()
    axes[1].hist(valid_sci, bins=20, color='steelblue', edgecolor='white')
    axes[1].axvline(SCI_THRESHOLD, color='r', ls='--', lw=1.5)
    axes[1].set_xlabel('SCI')
    axes[1].set_ylabel('通道数')
    axes[1].set_title('SCI 分布')

    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

for key, df in all_sci.items():
    plot_sci(df, key)

## 5. 坏道识别

坏道标准（满足任一条件即标记）：
- SCI < 0.75
- 信号全为 0 或常数
- 信号存在 Inf / NaN
- 信号峰峰值（P99-P1）< 阈值（近饱和或无信号）

In [ ]:
def identify_bad_channels(run, sci_df):
    d, ml = run['d'], run['ml']
    records = []
    for i in range(d.shape[1]):
        sig = d[:, i]
        src, det = ml[i, 0], ml[i, 1]
        wl_idx   = ml[i, 3]          # col3 = wavelength index
        wl_nm    = run['lambdas'][wl_idx - 1]

        has_inf  = not np.all(np.isfinite(sig))
        all_zero = np.all(sig == 0)
        finite_sig = sig[np.isfinite(sig)]
        pp = (np.nanpercentile(finite_sig, 99) - np.nanpercentile(finite_sig, 1)
              if len(finite_sig) > 0 and not all_zero else 0)

        row = sci_df[(sci_df['src'] == src) & (sci_df['det'] == det)]
        sci_val = float(row['SCI'].values[0]) if len(row) > 0 else np.nan
        low_sci = (sci_val < SCI_THRESHOLD) if np.isfinite(sci_val) else True

        reasons = []
        if has_inf:  reasons.append('Inf/NaN')
        if all_zero: reasons.append('全零')
        if pp == 0:  reasons.append('常数')
        if low_sci:  reasons.append(f'SCI={sci_val:.2f}')

        records.append({
            'ch_idx':   i,
            'label':    f"S{src}-D{det} {wl_nm}nm",
            'SCI':      sci_val,
            'peak_peak': pp,
            'has_inf':  has_inf,
            'is_bad':   len(reasons) > 0,
            'reason':   ', '.join(reasons) if reasons else 'OK',
        })
    return pd.DataFrame(records)

for sub, runs in datasets.items():
    for run in runs:
        key    = f"Sub-{sub} / {Path(run['filepath']).name}"
        sci_df = all_sci[key]
        bad_df = identify_bad_channels(run, sci_df)
        n_bad  = bad_df['is_bad'].sum()
        n_tot  = len(bad_df)
        print(f"\n{'='*60}")
        print(f"{key}:  坏道 {n_bad}/{n_tot}  ({100*n_bad/n_tot:.0f}%)")
        print(bad_df[bad_df['is_bad']][['label', 'SCI', 'reason']].to_string(index=False))

## 6. 运动伪影检测

使用**滑窗标准差**方法：窗口内信号的 Z-score 超过阈值的时刻标记为运动伪影。

In [ ]:
MA_Z_THRESH   = 20    # 基于原始强度信号的 z-score 阈值
MA_WIN_S      = 0.5   # 滑动窗口（秒）

def detect_motion_artifacts(run, bad_ch_df):
    d, t, ml = run['d'], run['t'], run['ml']
    sfreq = run['sfreq']
    win   = max(1, int(MA_WIN_S * sfreq))

    # 只用好道、第一波长
    good_wl1 = bad_ch_df[(~bad_ch_df['is_bad']) & 
                          (bad_ch_df['label'].str.contains(str(run['lambdas'][0])))]['ch_idx'].values

    if len(good_wl1) == 0:
        print("无可用好道，跳过运动伪影检测")
        return None, None

    # 差分（对数强度变化）
    d_good  = d[:, good_wl1]
    d_safe  = np.where(d_good > 0, d_good, np.nan)
    d_log   = np.log(d_safe)
    d_diff  = np.diff(d_log, axis=0)

    # 跨通道中位绝对偏差（MAD）
    ch_med  = np.nanmedian(d_diff, axis=1)
    ch_mad  = np.nanmedian(np.abs(d_diff - ch_med[:, None]), axis=1)
    z_score = np.abs(ch_med) / (ch_mad + 1e-10)

    ma_mask = z_score > MA_Z_THRESH
    ma_times = t[1:][ma_mask]

    return ma_times, z_score

fig_rows = sum(len(runs) for runs in datasets.values())
fig, axes = plt.subplots(fig_rows, 1, figsize=(16, 4 * fig_rows), squeeze=False)
ax_idx = 0

for sub, runs in datasets.items():
    for run in runs:
        key = f"Sub-{sub} / {Path(run['filepath']).name}"
        sci_df  = all_sci[key]
        bad_df  = identify_bad_channels(run, sci_df)
        ma_times, z_score = detect_motion_artifacts(run, bad_df)

        ax = axes[ax_idx, 0]
        if z_score is not None:
            ax.plot(run['t'][1:], z_score, lw=0.5, color='steelblue', label='Z-score')
            ax.axhline(MA_Z_THRESH, color='r', ls='--', lw=1.2, label=f'阈值={MA_Z_THRESH}')
            ax.fill_between(run['t'][1:], 0, z_score,
                            where=z_score > MA_Z_THRESH,
                            color='red', alpha=0.3, label=f'MA ({len(ma_times)} 点)')
            ax.set_title(f"{key} — 运动伪影 Z-score  ({len(ma_times)} 个可疑时刻)")
        else:
            ax.set_title(f"{key} — 无法检测")
        ax.set_xlabel("时间 (s)")
        ax.set_ylabel("Z-score")
        ax.legend(fontsize=8)
        ax_idx += 1

plt.tight_layout()
plt.show()

## 7. 辅助通道 (Aux) 查看

In [ ]:
for sub, runs in datasets.items():
    for run in runs:
        aux = run['aux']
        t   = run['t']
        key = f"Sub-{sub} / {Path(run['filepath']).name}"
        n_aux = aux.shape[1]

        fig, axes = plt.subplots(n_aux, 1, figsize=(16, 2 * n_aux), sharex=True)
        if n_aux == 1:
            axes = [axes]
        for i, ax in enumerate(axes):
            ax.plot(t, aux[:, i], lw=0.5, color='purple')
            ax.set_ylabel(f"Aux {i+1}")
        axes[-1].set_xlabel("时间 (s)")
        plt.suptitle(f"{key} — Aux 通道", fontsize=12)
        plt.tight_layout()
        plt.show()

## 8. QC 汇总表

In [ ]:
summary_rows = []
for sub, runs in datasets.items():
    for run in runs:
        key     = f"Sub-{sub} / {Path(run['filepath']).name}"
        sci_df  = all_sci[key]
        bad_df  = identify_bad_channels(run, sci_df)
        ma_times, _ = detect_motion_artifacts(run, bad_df)

        n_pairs   = len(sci_df)
        n_good_sci= sci_df['good'].sum()
        n_bad_ch  = bad_df['is_bad'].sum()
        n_total_ch= len(bad_df)
        ma_n      = len(ma_times) if ma_times is not None else 'N/A'
        ma_frac   = f"{len(ma_times)/len(run['t'])*100:.1f}%" \
                    if ma_times is not None else 'N/A'

        summary_rows.append({
            '被试': f"Sub-{sub}",
            '文件': Path(run['filepath']).name,
            '总时长(min)': f"{run['t'][-1]/60:.1f}",
            '好道/总道(SCI≥0.75)': f"{n_good_sci}/{n_pairs}",
            '坏道数(全类型)': f"{n_bad_ch}/{n_total_ch}",
            '运动伪影时刻数': ma_n,
            '运动伪影占比': ma_frac,
            '坏道列表(SCI)': ', '.join(sci_df[~sci_df['good']]['ch_label'].tolist()) or '无',
        })

summary_df = pd.DataFrame(summary_rows)
pd.set_option('display.max_colwidth', 80)
summary_df